In [43]:
# If running locally, make sure Ollama is installed and the model is pulled.
# Skip this cell if you're on Colab or using a remote Ollama endpoint.

import subprocess, shutil, sys

if shutil.which("ollama"):
    print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
else:
    print("⚠️ Ollama not found. Install from https://ollama.com/download")
    print("   Then run:  ollama pull llama3.1")

NAME                ID              SIZE      MODIFIED     
qwen2.5vl:latest    5ced39dfa4ba    6.0 GB    27 hours ago    
qwen3.5:4b          d8b0f5e9760c    3.3 GB    28 hours ago    



In [44]:
from __future__ import annotations
import json, statistics
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

print("imports ok")

imports ok


In [45]:
OLLAMA_MODEL    = "qwen3.5:4b"     # try: mistral, qwen2.5, llama3.2

In [46]:
class AnomalyModel(BaseModel):
    type: Literal[
        "readmission_30d",
        "date_conflict",
        "proc_diag_mismatch",
        "amount_under_10k",
        "duplicate_claim",
        "name_mismatch",
        "expired_id",
        "missing_mandatory_field",
        "uncovered_procedure",
        "blurry_scan",
        "frequency_anomaly",
        "provider_billing_pattern",
    ]
    severity: float = Field(ge=0.0, le=1.0)
    detail: str
    evidence: Dict[str, Any] = Field(default_factory=dict)


class FraudAssessment(BaseModel):
    fraud_score: float = Field(ge=0.0, le=1.0)
    anomalies: List[AnomalyModel]
    risk_level: Literal["LOW", "MEDIUM", "HIGH"]


print("schema ok")

schema ok


In [47]:
@dataclass
class Claim:
    claim_id: str
    patient_id: str
    provider_id: str
    patient_name: str
    subscriber_name: str
    service_code: str
    diagnosis_code: str
    amount: float
    service_date: datetime
    submitted_at: datetime
    admission_date: Optional[datetime] = None
    discharge_date: Optional[datetime] = None
    prescription_date: Optional[datetime] = None
    id_expiry_date: Optional[datetime] = None
    physician_signature: Optional[str] = None
    rendering_npi: Optional[str] = None
    blur_simulated: bool = False


class ClaimRepository:
    def __init__(self):
        self._claims: List[Claim] = []

    def load(self, claims: List[Claim]):
        self._claims = list(claims)

    def patient_claims(self, patient_id: str, lookback_days: int = 365) -> List[Claim]:
        cutoff = datetime.utcnow() - timedelta(days=lookback_days)
        return [c for c in self._claims
                if c.patient_id == patient_id and c.submitted_at >= cutoff]

    def provider_claims(self, provider_id: str, lookback_days: int = 90) -> List[Claim]:
        cutoff = datetime.utcnow() - timedelta(days=lookback_days)
        return [c for c in self._claims
                if c.provider_id == provider_id and c.submitted_at >= cutoff]

    def all_claims(self) -> List[Claim]:
        return list(self._claims)


REPO = ClaimRepository()
print("repo ok")

repo ok


In [48]:
# ---- C_001: readmission within 30 days ----
def detect_readmission_30d(claim: Claim, patient_history: List[Claim]) -> List[dict]:
    if not claim.admission_date:
        return []
    window_start = claim.admission_date - timedelta(days=30)
    hits = []
    for prior in patient_history:
        if prior.claim_id == claim.claim_id or not prior.discharge_date:
            continue
        if window_start <= prior.discharge_date < claim.admission_date:
            gap = (claim.admission_date - prior.discharge_date).days
            hits.append({
                "type": "readmission_30d",
                "severity": 0.85 if gap < 15 else 0.65,
                "detail": f"Readmission {gap}d after prior discharge ({prior.claim_id}).",
                "evidence": {"prior_claim_id": prior.claim_id, "gap_days": gap},
            })
    return hits


# ---- C_002: prescription after claim/discharge ----
def detect_date_conflict(claim: Claim, _h: List[Claim]) -> List[dict]:
    if not claim.prescription_date:
        return []
    anchor = claim.discharge_date or claim.service_date
    delta = (claim.prescription_date - anchor).days
    if delta >= 45:
        return [{
            "type": "date_conflict",
            "severity": min(0.6 + delta / 365.0, 1.0),
            "detail": f"Prescription is {delta}d after service/discharge ({anchor.date()}).",
            "evidence": {"prescription_date": claim.prescription_date.isoformat(),
                         "anchor_date": anchor.isoformat(), "delta_days": delta},
        }]
    return []


# ---- C_003: maternity CPT vs non-maternity dx ----
MATERNITY_CPTS = {"59400", "59409", "59410", "59510", "59610", "59618"}
MATERNITY_ICD_PREFIXES = ("O", "Z34", "Z37")
NON_MATERNITY_CHRONIC = ("E11", "I10", "E78", "J45")

def detect_proc_diag_mismatch(claim: Claim, _h: List[Claim]) -> List[dict]:
    if claim.service_code not in MATERNITY_CPTS:
        return []
    dx = claim.diagnosis_code.upper()
    if any(dx.startswith(p) for p in MATERNITY_ICD_PREFIXES):
        return []
    chronic = any(dx.startswith(p) for p in NON_MATERNITY_CHRONIC)
    return [{
        "type": "proc_diag_mismatch",
        "severity": 0.9 if chronic else 0.75,
        "detail": f"Maternity CPT {claim.service_code} with non-maternity dx {claim.diagnosis_code}.",
        "evidence": {"cpt": claim.service_code, "icd": claim.diagnosis_code},
    }]


# ---- C_004: amount just under $10k ----
REVIEW_THRESHOLD = 10_000.0
STRUCTURING_BAND = 500.0

def detect_amount_structuring(claim: Claim, _h: List[Claim]) -> List[dict]:
    delta = REVIEW_THRESHOLD - claim.amount
    if 0 < delta <= STRUCTURING_BAND:
        sev = 0.9 - (delta / STRUCTURING_BAND) * 0.3
        return [{
            "type": "amount_under_10k",
            "severity": round(sev, 3),
            "detail": f"Billed ${claim.amount:,.2f} — ${delta:,.2f} below ${REVIEW_THRESHOLD:,.0f} threshold.",
            "evidence": {"amount": claim.amount, "delta": delta},
        }]
    return []


# ---- C_005: duplicate claim no + service date ----
def detect_duplicate_claim(claim: Claim, _h: List[Claim]) -> List[dict]:
    dupes = [
        c for c in REPO.all_claims()
        if c is not claim
        and c.claim_id == claim.claim_id
        and c.service_date == claim.service_date
    ]
    if dupes:
        return [{
            "type": "duplicate_claim",
            "severity": 1.0,
            "detail": f"Claim {claim.claim_id} resubmitted on identical service date.",
            "evidence": {"matching_files": len(dupes) + 1},
        }]
    return []


# ---- C_006: name mismatch ----
def _norm(n: str) -> str:
    return " ".join((n or "").lower().strip().split())

def detect_name_mismatch(claim: Claim, _h: List[Claim]) -> List[dict]:
    if not claim.patient_name or not claim.subscriber_name:
        return []
    if _norm(claim.patient_name) != _norm(claim.subscriber_name):
        return [{
            "type": "name_mismatch",
            "severity": 0.8,
            "detail": f"Patient '{claim.patient_name}' != subscriber '{claim.subscriber_name}'.",
            "evidence": {"patient_name": claim.patient_name,
                         "subscriber_name": claim.subscriber_name},
        }]
    return []


# ---- 4.1: expired ID ----
def detect_expired_id(claim: Claim, _h: List[Claim]) -> List[dict]:
    if not claim.id_expiry_date:
        return []
    if claim.id_expiry_date < claim.service_date:
        days = (claim.service_date - claim.id_expiry_date).days
        return [{
            "type": "expired_id",
            "severity": 0.6,
            "detail": f"ID expired {days}d before service date.",
            "evidence": {"expiry": claim.id_expiry_date.isoformat(),
                         "service_date": claim.service_date.isoformat()},
        }]
    return []


# ---- 4.2: missing mandatory fields ----
def detect_missing_fields(claim: Claim, _h: List[Claim]) -> List[dict]:
    missing = []
    if not claim.physician_signature: missing.append("physician_signature")
    if not claim.rendering_npi:       missing.append("rendering_npi")
    if missing:
        return [{
            "type": "missing_mandatory_field",
            "severity": 0.55,
            "detail": f"Missing: {', '.join(missing)}.",
            "evidence": {"missing": missing},
        }]
    return []


# ---- 4.3: uncovered/cosmetic procedure ----
UNCOVERED_CPTS = {
    "17000": "Destruction of premalignant lesions (cosmetic)",
    "21120": "Genioplasty (cosmetic craniofacial)",
}

def detect_uncovered_procedure(claim: Claim, _h: List[Claim]) -> List[dict]:
    if claim.service_code in UNCOVERED_CPTS:
        return [{
            "type": "uncovered_procedure",
            "severity": 0.5,
            "detail": f"CPT {claim.service_code}: {UNCOVERED_CPTS[claim.service_code]}.",
            "evidence": {"cpt": claim.service_code},
        }]
    return []


# ---- 4.4: blurry scan ----
def detect_blurry_scan(claim: Claim, _h: List[Claim]) -> List[dict]:
    if claim.blur_simulated:
        return [{
            "type": "blurry_scan",
            "severity": 0.4,
            "detail": "Document flagged as blurry/low-quality scan.",
            "evidence": {"blur_simulated": True},
        }]
    return []


# ---- statistical baselines ----
def detect_frequency_anomaly(claim: Claim, patient_history: List[Claim]) -> List[dict]:
    if len(patient_history) < 5:
        return []
    buckets: Dict[str, int] = {}
    for c in patient_history:
        k = c.submitted_at.strftime("%Y-%m")
        buckets[k] = buckets.get(k, 0) + 1
    if len(buckets) < 3:
        return []
    counts = list(buckets.values())
    mean = statistics.mean(counts)
    std = statistics.pstdev(counts) or 1.0
    cutoff = datetime.utcnow() - timedelta(days=30)
    recent = sum(1 for c in patient_history if c.submitted_at >= cutoff)
    z = (recent - mean) / std
    if z >= 2.5:
        return [{
            "type": "frequency_anomaly",
            "severity": min(z / 5.0, 1.0),
            "detail": f"{recent} claims/30d vs baseline {mean:.1f} (z={z:.2f}).",
            "evidence": {"recent": recent, "mean": mean, "z": z},
        }]
    return []


def detect_provider_pattern(claim: Claim, provider_history: List[Claim]) -> List[dict]:
    if len(provider_history) < 10:
        return []
    amounts = [c.amount for c in provider_history]
    mean = statistics.mean(amounts)
    std = statistics.pstdev(amounts) or 1.0
    z = (claim.amount - mean) / std
    if abs(z) >= 2.5:
        return [{
            "type": "provider_billing_pattern",
            "severity": min(abs(z) / 5.0, 1.0),
            "detail": f"${claim.amount:,.2f} vs provider mean ${mean:,.2f} (z={z:.2f}).",
            "evidence": {"amount": claim.amount, "mean": mean, "z": z},
        }]
    return []


print("detectors ok")

detectors ok


In [49]:
WEIGHTS = {
    "duplicate_claim":          1.00,
    "readmission_30d":          0.85,
    "date_conflict":            0.85,
    "proc_diag_mismatch":       0.90,
    "amount_under_10k":         0.80,
    "name_mismatch":            0.80,
    "expired_id":               0.60,
    "missing_mandatory_field":  0.55,
    "uncovered_procedure":      0.45,
    "blurry_scan":              0.30,
    "frequency_anomaly":        0.25,
    "provider_billing_pattern": 0.25,
}

def composite_score(anomalies: List[dict]) -> float:
    if not anomalies:
        return 0.0
    total = sum(WEIGHTS.get(a["type"], 0.25) * a["severity"] for a in anomalies)
    return round(min(total, 1.0), 4)

def risk_level(score: float) -> str:
    if score >= 0.6:  return "HIGH"
    if score >= 0.3:  return "MEDIUM"
    return "LOW"

print("scoring ok")

scoring ok


In [50]:
def _dt(v: Any) -> Optional[datetime]:
    if v is None or v == "": return None
    if isinstance(v, datetime): return v
    try:
        return datetime.fromisoformat(str(v).replace("Z", "+00:00")).replace(tzinfo=None)
    except ValueError:
        return None


@tool
def fetch_patient_history(patient_id: str, lookback_days: int = 365) -> list:
    """Fetch prior claims for a patient (frequency, service codes, dates)."""
    return [{
        "claim_id": c.claim_id, "provider_id": c.provider_id,
        "service_code": c.service_code, "amount": c.amount,
        "service_date": c.service_date.isoformat(),
        "admission_date": c.admission_date.isoformat() if c.admission_date else None,
        "discharge_date": c.discharge_date.isoformat() if c.discharge_date else None,
        "submitted_at": c.submitted_at.isoformat(),
    } for c in REPO.patient_claims(patient_id, lookback_days)]


@tool
def fetch_provider_history(provider_id: str, lookback_days: int = 90) -> list:
    """Fetch recent claims billed by a provider to establish a baseline."""
    return [{
        "claim_id": c.claim_id, "patient_id": c.patient_id,
        "service_code": c.service_code, "amount": c.amount,
        "submitted_at": c.submitted_at.isoformat(),
    } for c in REPO.provider_claims(provider_id, lookback_days)]


@tool
def run_detectors(claim_json: dict) -> dict:
    """Run every fraud detector against a claim.

    Args:
        claim_json: dict with keys claim_id, patient_id, provider_id,
            patient_name, subscriber_name, service_code, diagnosis_code,
            amount, service_date (ISO), submitted_at (ISO). Optional:
            admission_date, discharge_date, prescription_date,
            id_expiry_date, physician_signature, rendering_npi, blur_simulated.
    """
    c = Claim(
        claim_id=claim_json["claim_id"],
        patient_id=claim_json["patient_id"],
        provider_id=claim_json["provider_id"],
        patient_name=claim_json.get("patient_name", ""),
        subscriber_name=claim_json.get("subscriber_name", ""),
        service_code=claim_json["service_code"],
        diagnosis_code=claim_json.get("diagnosis_code", ""),
        amount=float(claim_json["amount"]),
        service_date=_dt(claim_json["service_date"]) or datetime.utcnow(),
        submitted_at=_dt(claim_json["submitted_at"]) or datetime.utcnow(),
        admission_date=_dt(claim_json.get("admission_date")),
        discharge_date=_dt(claim_json.get("discharge_date")),
        prescription_date=_dt(claim_json.get("prescription_date")),
        id_expiry_date=_dt(claim_json.get("id_expiry_date")),
        physician_signature=claim_json.get("physician_signature"),
        rendering_npi=claim_json.get("rendering_npi"),
        blur_simulated=bool(claim_json.get("blur_simulated", False)),
    )

    ph = REPO.patient_claims(c.patient_id)
    prh = REPO.provider_claims(c.provider_id)

    anomalies = []
    anomalies += detect_readmission_30d(c, ph)
    anomalies += detect_date_conflict(c, ph)
    anomalies += detect_proc_diag_mismatch(c, ph)
    anomalies += detect_amount_structuring(c, ph)
    anomalies += detect_duplicate_claim(c, ph)
    anomalies += detect_name_mismatch(c, ph)
    anomalies += detect_expired_id(c, ph)
    anomalies += detect_missing_fields(c, ph)
    anomalies += detect_uncovered_procedure(c, ph)
    anomalies += detect_blurry_scan(c, ph)
    anomalies += detect_frequency_anomaly(c, ph)
    anomalies += detect_provider_pattern(c, prh)

    score = composite_score(anomalies)
    return {"anomalies": anomalies, "composite_score": score,
            "risk_level": risk_level(score)}


llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

fraud_agent = create_agent(
    model=llm,
    tools=[fetch_patient_history, fetch_provider_history, run_detectors],
    system_prompt=(
        "You are a healthcare fraud detection analyst. For every claim:\n"
        "1. Call fetch_patient_history(patient_id).\n"
        "2. Call fetch_provider_history(provider_id).\n"
        "3. Call run_detectors with the full claim payload.\n"
        "4. Return a FraudAssessment using ONLY run_detectors output. "
        "fraud_score = composite_score; risk_level = risk_level; "
        "anomalies = anomalies. Do NOT invent or drop anomalies."
    ),
    response_format=ToolStrategy(FraudAssessment),
)

print("agent ready")

agent ready


In [51]:
now = datetime.utcnow()

seed_claims = [
    # C_001 readmission prior
    Claim("C_001_prior", "PT_19116", "PRV_77", "Alice Ng", "Alice Ng",
          "99223", "J18.9", 12000.0,
          service_date=now - timedelta(days=25),
          submitted_at=now - timedelta(days=25),
          admission_date=now - timedelta(days=25),
          discharge_date=now - timedelta(days=10)),

    # C_005 duplicate pair
    Claim("C_005_DUP", "PT_20322", "PRV_42", "Bob Ray", "Bob Ray",
          "99213", "E11.9", 300.0,
          service_date=now - timedelta(days=5),
          submitted_at=now - timedelta(days=5)),
    Claim("C_005_DUP", "PT_20322", "PRV_42", "Bob Ray", "Bob Ray",
          "99213", "E11.9", 300.0,
          service_date=now - timedelta(days=5),
          submitted_at=now - timedelta(days=4)),

    # provider baseline for PRV_42
    *[Claim(f"BASE_{i}", f"PT_{i:05d}", "PRV_42", "x", "x",
            "99213", "Z00.00", 150.0 + i,
            service_date=now - timedelta(days=i*3),
            submitted_at=now - timedelta(days=i*3))
      for i in range(1, 13)],
]

REPO.load(seed_claims)
print(f"loaded {len(seed_claims)} claims")

loaded 15 claims


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_804\625898524.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [52]:
now = datetime.utcnow()

DEMO_CLAIMS = {
    "C_001_readmission": {
        "claim_id": "C_001_NEW", "patient_id": "PT_19116", "provider_id": "PRV_77",
        "patient_name": "Alice Ng", "subscriber_name": "Alice Ng",
        "service_code": "99223", "diagnosis_code": "J18.9", "amount": 12000.0,
        "admission_date": now.isoformat(),
        "service_date": now.isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_002_date_conflict": {
        "claim_id": "C_002_NEW", "patient_id": "PT_99733", "provider_id": "PRV_10",
        "patient_name": "Cara Vo", "subscriber_name": "Cara Vo",
        "service_code": "99214", "diagnosis_code": "I10", "amount": 400.0,
        "service_date": (now - timedelta(days=60)).isoformat(),
        "discharge_date": (now - timedelta(days=60)).isoformat(),
        "prescription_date": (now - timedelta(days=10)).isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_003_proc_diag_mismatch": {
        "claim_id": "C_003_NEW", "patient_id": "PT_62350", "provider_id": "PRV_21",
        "patient_name": "Dana Kim", "subscriber_name": "Dana Kim",
        "service_code": "59400", "diagnosis_code": "E11.9", "amount": 8500.0,
        "service_date": (now - timedelta(days=3)).isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_004_amount_structuring": {
        "claim_id": "C_004_NEW", "patient_id": "PT_69470", "provider_id": "PRV_33",
        "patient_name": "Evan Lo", "subscriber_name": "Evan Lo",
        "service_code": "99215", "diagnosis_code": "I10", "amount": 9875.00,
        "service_date": (now - timedelta(days=2)).isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_005_duplicate": {
        "claim_id": "C_005_DUP", "patient_id": "PT_20322", "provider_id": "PRV_42",
        "patient_name": "Bob Ray", "subscriber_name": "Bob Ray",
        "service_code": "99213", "diagnosis_code": "E11.9", "amount": 300.0,
        "service_date": (now - timedelta(days=5)).isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_006_name_mismatch": {
        "claim_id": "C_006_NEW", "patient_id": "PT_39451", "provider_id": "PRV_55",
        "patient_name": "Fiona Marsh", "subscriber_name": "Frank Marsh",
        "service_code": "99213", "diagnosis_code": "J45", "amount": 220.0,
        "service_date": (now - timedelta(days=1)).isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_010_expired_id": {
        "claim_id": "C_010_NEW", "patient_id": "PT_20745", "provider_id": "PRV_60",
        "patient_name": "Gina Pell", "subscriber_name": "Gina Pell",
        "service_code": "99213", "diagnosis_code": "Z00.00", "amount": 180.0,
        "service_date": now.isoformat(),
        "id_expiry_date": (now - timedelta(days=90)).isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_008_missing_fields": {
        "claim_id": "C_008_NEW", "patient_id": "PT_82132", "provider_id": "PRV_61",
        "patient_name": "Hal Ortiz", "subscriber_name": "Hal Ortiz",
        "service_code": "99214", "diagnosis_code": "I10", "amount": 260.0,
        "service_date": now.isoformat(),
        "submitted_at": now.isoformat(),
    },
    "C_007_uncovered": {
        "claim_id": "C_007_NEW", "patient_id": "PT_71993", "provider_id": "PRV_62",
        "patient_name": "Ivy Chen", "subscriber_name": "Ivy Chen",
        "service_code": "17000", "diagnosis_code": "L82.1", "amount": 450.0,
        "service_date": now.isoformat(),
        "submitted_at": now.isoformat(),
    },
    "blurry_scan": {
        "claim_id": "C_040_NEW", "patient_id": "PT_90000", "provider_id": "PRV_63",
        "patient_name": "Jon Blake", "subscriber_name": "Jon Blake",
        "service_code": "99213", "diagnosis_code": "Z00.00", "amount": 175.0,
        "service_date": now.isoformat(),
        "submitted_at": now.isoformat(),
        "blur_simulated": True,
    },
}

print(f"prepared {len(DEMO_CLAIMS)} test claims")

prepared 10 test claims


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_804\321065296.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [53]:
label = "C_003_proc_diag_mismatch"
payload = DEMO_CLAIMS[label]

result = fraud_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            f"Assess this claim for fraud. Pass every field into run_detectors.\n"
            f"claim_json = {json.dumps(payload)}"
        ),
    }]
})

assessment: FraudAssessment = result["structured_response"]
print(f"\n=== {label} ===")
print(assessment.model_dump_json(indent=2))

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_804\1453514370.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.utcnow() - timedelta(days=lookback_days)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_804\1453514370.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.utcnow() - timedelta(days=lookback_days)



=== C_003_proc_diag_mismatch ===
{
  "fraud_score": 1.0,
  "anomalies": [
    {
      "type": "proc_diag_mismatch",
      "severity": 0.9,
      "detail": "Maternity CPT 59400 with non-maternity dx E11.9.",
      "evidence": {
        "cpt": "59400",
        "icd": "E11.9"
      }
    },
    {
      "type": "missing_mandatory_field",
      "severity": 0.55,
      "detail": "Missing: physician_signature, rendering_npi.",
      "evidence": {
        "missing": [
          "physician_signature",
          "rendering_npi"
        ]
      }
    }
  ],
  "risk_level": "HIGH"
}
